# Lab 4: Controlled Diffusion, CLIP, and LLM Watermarking
**EECE.4860/5860 Introduction to Generative AI, Spring 2027**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acanets/genai-labs/blob/spring27/labs/lab4/lab4.ipynb)

* **Part A (all students):** steer Stable Diffusion 1.5 with a **ControlNet** edge map, sweep guidance scale and denoising steps, and score each image with your own **CLIPScore** implementation.
* **Part B (all students):** **zero-shot image classification** with CLIP, and the effect of prompt templates.
* **Part C (EECE.5860 only):** implement the **green-list watermark** for LLM text (Kirchenbauer et al., 2023), detect it with a z-test, and attack it by editing tokens.

**Before you start:** *File > Save a copy in Drive*, then *Runtime > Change runtime type > T4 GPU*.

**Time budget on a Colab T4:** about 10 min for Part A, 3 min for Part B, and 5 min for Part C. Models: Stable Diffusion 1.5 + ControlNet-Canny (CreativeML OpenRAIL-M; the safety checker stays on), CLIP ViT-B/32, Qwen3-0.6B.

**Grading (15 points):** 2 for a notebook that runs top to bottom, 8 for the `[CHECK ...]` lines, and 5 for the report. EECE.5860: Part C adds 5 points and the total is scaled to 15. Submit the `.ipynb` with outputs and a PDF of it on Canvas.

## 0. Setup

In [ ]:
STUDENT_ID_LAST4 = ""   # last 4 digits of your student ID, e.g. "0427"
SECTION = "4860"        # "4860" or "5860"

In [ ]:
import importlib.util, os
if importlib.util.find_spec("genai_utils") is None:
    get_ipython().run_line_magic("pip", 'install -q "git+https://github.com/acanets/genai-labs@spring27"')
if importlib.util.find_spec("diffusers") is None:
    get_ipython().run_line_magic("pip", "install -q diffusers")

import math, random, time
import numpy as np
import torch
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from transformers import CLIPModel, CLIPProcessor
from genai_utils import setup_env, Checker, variant, measure

QUICK = os.environ.get("GENAI_QUICK") == "1"   # instructor smoke test only; leave as is
device, dtype = setup_env(seed=int(STUDENT_ID_LAST4 or 0))
ck = Checker("lab4", STUDENT_ID_LAST4, SECTION)
V = variant(STUDENT_ID_LAST4, 4)
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

## Part A: Controlled generation with ControlNet (all students)

### A1. Your CLIPScore
CLIPScore (Hessel et al., 2021) measures how well an image matches a text without a reference image: $\text{CLIPScore}(c, v) = w \cdot \max(\cos(c, v), 0)$ with $w = 2.5$, where $v$ and $c$ are CLIP's image and text embeddings.

**Task:** implement `clip_score(images, texts)` for paired lists. Use `clip_proc(text=..., images=..., return_tensors="pt", padding=True)`, `clip.get_image_features(pixel_values=...)` and `clip.get_text_features(input_ids=..., attention_mask=...)`, L2-normalize both, and return a list of floats.

In [ ]:
def as_tensor(x):
    """CLIP feature calls return a tensor in some transformers versions and an output object in others."""
    return x if isinstance(x, torch.Tensor) else x.pooler_output

@torch.no_grad()
def clip_score(images, texts, w=2.5):
    # TODO: your code here
    raise NotImplementedError

def shape_image(kind, color, size=224):
    im = Image.new("RGB", (size, size), "white")
    d = ImageDraw.Draw(im)
    box = (40, 40, size - 40, size - 40)
    (d.ellipse if kind == "circle" else d.rectangle)(box, fill=color)
    return im

red_circle = shape_image("circle", "red")
s_right, s_wrong = clip_score([red_circle, red_circle], ["a red circle", "a blue square"])
print(f"CLIPScore red circle vs 'a red circle': {s_right:.3f}; vs 'a blue square': {s_wrong:.3f}")
ck.expect_true("A.clip_score", 0 <= s_wrong < s_right <= 2.5, value=(round(s_right, 3), round(s_wrong, 3)))

### A2. A conditioning image
ControlNet (Zhang et al., 2023) adds spatial control to a frozen text-to-image model: a trainable copy of its encoder reads a condition image (here, Canny edges) and is connected through zero-initialized convolutions. We draw a simple house scene and extract its edges. *Optional:* upload your own photo and set `MY_IMAGE` to its path.

In [ ]:
import cv2
MY_IMAGE = None   # e.g. "/content/my_photo.jpg"
RES = 256 if QUICK else 512

def scene(size=512):
    im = Image.new("RGB", (size, size), "white")
    d = ImageDraw.Draw(im)
    d.rectangle((140, 260, 360, 440), outline="black", width=6)                  # house
    d.polygon([(120, 260), (250, 140), (380, 260)], outline="black", width=6)    # roof
    d.rectangle((225, 350, 275, 440), outline="black", width=5)                  # door
    d.ellipse((400, 60, 480, 140), outline="black", width=5)                     # sun
    d.rectangle((60, 360, 80, 440), outline="black", width=5)                    # tree trunk
    d.ellipse((20, 250, 120, 370), outline="black", width=5)                     # tree crown
    d.line((0, 440, size, 440), fill="black", width=5)                           # ground
    return im

src = Image.open(MY_IMAGE).convert("RGB").resize((RES, RES)) if MY_IMAGE else scene().resize((RES, RES))
edges = cv2.Canny(np.array(src), 100, 200)
control = Image.fromarray(np.stack([edges] * 3, axis=-1))
PROMPTS = ["a cozy cottage in a snowy forest, watercolor painting",
           "a futuristic house on Mars, digital art",
           "a beach house at sunset, oil painting",
           "a haunted house at night, comic book style"]
PROMPT = PROMPTS[V]
print("Your prompt:", PROMPT)
fig, ax = plt.subplots(1, 2, figsize=(6, 3)); ax[0].imshow(src); ax[1].imshow(control)
for a in ax: a.axis("off")
plt.show()

### A3. Sweep guidance scale and steps
We generate 9 images with the **same seed**: guidance scale ∈ {3, 7.5, 12} × denoising steps ∈ {10, 20, 40}. For each image we record CLIPScore and seconds per image. Then two more images vary `controlnet_conditioning_scale` (how strongly the edges constrain the image).

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
SD = "stable-diffusion-v1-5/stable-diffusion-v1-5"
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=dtype)
pipe = StableDiffusionControlNetPipeline.from_pretrained(SD, controlnet=controlnet, torch_dtype=dtype).to(device)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

GUIDANCE = [3.0, 7.5, 12.0]
STEPS = [2, 3, 4] if QUICK else [10, 20, 40]
rows, images = [], {}
with measure("A3 sweep"):
    for g in GUIDANCE:
        for s in STEPS:
            gen = torch.Generator("cpu").manual_seed(1000 + V)
            t0 = time.perf_counter()
            img = pipe(PROMPT, image=control, num_inference_steps=s, guidance_scale=g, generator=gen,
                       height=RES, width=RES).images[0]
            secs = time.perf_counter() - t0
            cs = clip_score([img], [PROMPT])[0]
            rows.append({"guidance": g, "steps": s, "clipscore": round(cs, 4), "seconds": round(secs, 2)})
            images[(g, s)] = img
fig, ax = plt.subplots(3, 3, figsize=(9, 9))
for i, g in enumerate(GUIDANCE):
    for j, s in enumerate(STEPS):
        ax[i, j].imshow(images[(g, s)]); ax[i, j].axis("off")
        ax[i, j].set_title(f"g={g}, steps={s}\nCLIPScore={[r for r in rows if r['guidance']==g and r['steps']==s][0]['clipscore']:.3f}", fontsize=8)
plt.tight_layout(); plt.show()
for r in rows:
    print(r)
ck.record("A.sweep", rows)
ck.expect_true("A.sweep_complete", len(rows) == 9 and all(0 <= r["clipscore"] <= 2.5 for r in rows))
t_lo = np.mean([r["seconds"] for r in rows if r["steps"] == STEPS[0]])
t_hi = np.mean([r["seconds"] for r in rows if r["steps"] == STEPS[-1]])
ck.expect_true("A.more_steps_cost_more_time", t_hi > t_lo, value=(round(t_lo, 2), round(t_hi, 2)))

In [ ]:
cond_imgs = []
for scale in (0.3, 1.0):
    gen = torch.Generator("cpu").manual_seed(1000 + V)
    cond_imgs.append(pipe(PROMPT, image=control, num_inference_steps=STEPS[1], guidance_scale=7.5, generator=gen,
                          controlnet_conditioning_scale=scale, height=RES, width=RES).images[0])
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, im, t in zip(ax, [control] + cond_imgs, ["edges", "conditioning scale 0.3", "conditioning scale 1.0"]):
    a.imshow(im); a.set_title(t, fontsize=9); a.axis("off")
plt.show()
del pipe, controlnet
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Part B: Zero-shot classification with CLIP (all students)
CLIP classifies an image by comparing its embedding with the embeddings of one text prompt per class and picking the most similar (Radford et al., 2021). We use 200 CIFAR-10 test images, chosen by your variant.

**Task:** implement `zero_shot_predict(images, class_names, templates)`. For each class, embed every template filled with the class name (e.g. `"a photo of a {}."`), L2-normalize, **average over templates**, and re-normalize. Then predict the class whose text embedding has the highest cosine similarity with each image embedding. Return a list of class indices.

In [ ]:
from datasets import load_dataset
cifar = load_dataset("uoft-cs/cifar10", split="test")
CLASSES = cifar.features["label"].names
idx = random.Random(V).sample(range(len(cifar)), 20 if QUICK else 200)
imgs = [cifar[i]["img"].convert("RGB") for i in idx]
labels = [cifar[i]["label"] for i in idx]

@torch.no_grad()
def zero_shot_predict(images, class_names, templates, batch_size=64):
    # TODO: your code here
    raise NotImplementedError

TEMPLATE_SETS = {
    "{}": ["{}"],
    "a photo of a {}.": ["a photo of a {}."],
    "ensemble of 3": ["a photo of a {}.", "a blurry photo of a {}.", "a low resolution photo of a {}."],
}
acc = {}
with measure("B zero-shot"):
    for name, tpls in TEMPLATE_SETS.items():
        p = zero_shot_predict(imgs, CLASSES, tpls)
        acc[name] = round(sum(int(a == b) for a, b in zip(p, labels)) / len(labels), 3)
        print(f"{name:20s} accuracy = {acc[name]:.3f}")
ck.record("B.accuracy", acc)
ck.expect_range("B.zero_shot_accuracy", max(acc.values()), 0.7, 1.0)

## Part C: Watermarking LLM text (EECE.5860 only; EECE.4860 students may skip to the report)
Kirchenbauer et al. (2023): at each step, the previous token seeds a random split of the vocabulary into a **green list** (fraction $\gamma$) and a red list; the generator adds $\delta$ to the logits of green tokens. A detector that knows the key counts green tokens and computes
$z = (|s|_G - \gamma T)/\sqrt{T\gamma(1-\gamma)}$. Unwatermarked text has $z \approx 0$; watermarked text has a large $z$.

**Tasks:** implement `green_list` (exactly `int(gamma * vocab_size)` green tokens, deterministic given `prev_token` and `key`), the logits processor's `__call__`, and `z_score`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor, LogitsProcessorList
GAMMA, DELTA, KEY = 0.25, 2.0, 15485863
_cache = {}

def green_list(prev_token, vocab_size, gamma=GAMMA, key=KEY):
    """Boolean tensor (CPU) of length vocab_size with exactly int(gamma*vocab_size) True entries."""
    ck_ = (int(prev_token), vocab_size, gamma, key)
    if ck_ in _cache:
        return _cache[ck_]
    # TODO: your code here
    raise NotImplementedError
    _cache[ck_] = mask
    return mask

class GreenListWatermark(LogitsProcessor):
    def __init__(self, gamma=GAMMA, delta=DELTA):
        self.gamma, self.delta = gamma, delta

    def __call__(self, input_ids, scores):
        # TODO: your code here
        raise NotImplementedError

def z_score(prev_last_token, tokens, vocab_size, gamma=GAMMA):
    """z-statistic for a generated token list; the first token's predecessor is prev_last_token."""
    # TODO: your code here
    raise NotImplementedError

if SECTION == "5860":
    m1, m2, m3 = green_list(5, 1000), green_list(5, 1000), green_list(6, 1000)
    ck.expect_true("C.green_list", int(m1.sum()) == 250 and torch.equal(m1, m2) and not torch.equal(m1, m3))
    def _chain(T, want_green, V=1000):   # T tokens, each green (or red) with respect to its predecessor
        prev, seq = 0, []
        for _ in range(T):
            t = int(torch.nonzero(green_list(prev, V) == want_green)[0])
            seq.append(t); prev = t
        return seq
    ck.expect_fn("C.z_score", lambda seq: round(z_score(0, seq, 1000), 3), [
        ((_chain(40, True),), round(30 / math.sqrt(7.5), 3)),    # all green: (40 - 10) / sqrt(40 * 0.25 * 0.75)
        ((_chain(40, False),), round(-10 / math.sqrt(7.5), 3)),  # all red
    ], tol=1e-3)

In [ ]:
if SECTION == "5860":
    LM = "Qwen/Qwen3-0.6B"
    tok = AutoTokenizer.from_pretrained(LM); tok.padding_side = "left"
    lm = AutoModelForCausalLM.from_pretrained(LM, dtype=dtype).to(device).eval()
    topics = ["the history of the bicycle", "how bees make honey", "why the sky is blue", "the rules of chess",
              "how vaccines work", "the water cycle", "how a refrigerator works", "the life of a star"][: 2 if QUICK else 8]
    prompts = [tok.apply_chat_template([{"role": "user", "content": f"Write a paragraph about {t}."}], tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False) for t in topics]
    enc = tok(prompts, return_tensors="pt", padding=True).to(device)
    VOCAB = lm.get_output_embeddings().weight.shape[0]
    N_NEW = 40 if QUICK else 150

    @torch.no_grad()
    def gen(processors):
        torch.manual_seed(V)
        out = lm.generate(**enc, max_new_tokens=N_NEW, min_new_tokens=N_NEW, do_sample=True, temperature=0.7, top_p=1.0, top_k=0,
                          logits_processor=processors, pad_token_id=tok.pad_token_id)
        return [row.tolist() for row in out[:, enc["input_ids"].shape[1]:]]

    with measure("C generate"):
        wm_tokens = gen(LogitsProcessorList([GreenListWatermark()]))
        plain_tokens = gen(LogitsProcessorList([]))
    last = enc["input_ids"][:, -1].tolist()
    z_wm = [z_score(l, t, VOCAB) for l, t in zip(last, wm_tokens)]
    z_plain = [z_score(l, t, VOCAB) for l, t in zip(last, plain_tokens)]
    print("watermarked sample:", tok.decode(wm_tokens[0], skip_special_tokens=True)[:300])
    print(f"mean z: watermarked {np.mean(z_wm):.2f}, plain {np.mean(z_plain):.2f}")
    rng = random.Random(V)
    attack = {}
    for p in (0.0, 0.1, 0.3, 0.5):
        zs = []
        for l, t in zip(last, wm_tokens):
            t2 = [rng.randrange(VOCAB) if rng.random() < p else x for x in t]
            zs.append(z_score(l, t2, VOCAB))
        attack[p] = round(float(np.mean(zs)), 2)
    print("mean z after replacing a fraction of tokens:", attack)
    ck.record("C.z", {"watermarked": round(float(np.mean(z_wm)), 2), "plain": round(float(np.mean(z_plain)), 2), "attack": attack})
    ck.expect_true("C.watermark_detected", np.mean(z_wm) > 4 and np.mean(z_plain) < 2,
                   value=(round(float(np.mean(z_wm)), 2), round(float(np.mean(z_plain)), 2)))
    ck.expect_true("C.edits_weaken_watermark", attack[0.5] < attack[0.0], value=attack)

## Results summary
Run this cell last. **Do not edit its output.**

In [ ]:
_ = ck.summary()

## Report (5 points)
Answer in the Markdown cells below (2–5 sentences each), citing your own numbers and images.

### 1. Summary
**Your answer:**

### 2. Problems encountered and how you solved them
**Your answer:**

### 3. Questions
**Q1 (A3).** How did the guidance scale change the images and the CLIPScore? Relate this to how classifier-free guidance combines conditional and unconditional predictions (Ho & Salimans, 2022).

**Your answer:**

**Q2 (A3).** How did the number of denoising steps affect quality and time? Which setting would you choose for an interactive app, and why?

**Your answer:**

**Q3 (A3).** Compare conditioning scale 0.3 with 1.0. What does the ControlNet branch contribute, and why is the original model kept frozen?

**Your answer:**

**Q4 (B).** Report zero-shot accuracy for the three template sets. Why might "a photo of a {}." beat the bare class name?

**Your answer:**

**Q5 (C, EECE.5860).** Report the mean z for watermarked and plain text and how it changed as you replaced tokens. Why does a longer text make detection more reliable?

**Your answer:**

**Q6 (C, EECE.5860).** Name one realistic attack on this watermark that is harder to defend against than random token replacement, and one limitation of watermarking as a policy tool (see Lecture 13).

**Your answer:**

### 4. Lessons learned
**Your answer:**

### 5. AI-use disclosure
*Follow the course AI-use policy in the syllabus; write "None" if you used no AI tools.*

**Your answer:**